<a href="https://colab.research.google.com/github/aflahzaki/ProposalSeminarS6/blob/main/NGBoostDiCE_WaterQuality_Canada_Proof_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NGBoost + DiCE: Water Quality Classification (Multi-class)
## Dataset: Canada Water Quality (CCME WQI - 5 Classes)
## Pipeline V2 - Preprocessing mengikuti Al Bataineh et al. (2026)

**Konteks:**
Dataset Canada Water Quality menggunakan Canadian Council of Ministers of the Environment Water Quality Index (CCME WQI) dengan 5 kelas: Excellent, Good, Fair, Marginal, Poor. Dataset memiliki 3949 sampel dengan 8 fitur numerik dan tidak memiliki missing values.

**Alur Preprocessing (Al Bataineh et al., 2026 - Algorithm 3):**
1. Handle missing values using median imputation (seluruh data - untuk konsistensi pipeline)
2. Normalize all features to range [0,1] using MinMaxScaler (seluruh data)
3. Split D into training set (70%), validation set (15%), and test set (15%)

**DiCE Binary Grouping:**
- Training: 5 kelas (Excellent, Good, Fair, Marginal, Poor) menggunakan k_categorical(5)
- Counterfactual: Binary grouping {Marginal, Poor} = "Unacceptable", {Excellent, Good, Fair} = "Acceptable"
- Target: Unacceptable -> Acceptable

**Referensi Pendukung:**
- Al Bataineh et al. (2026) - Algorithm 3: Impute, Normalize, Split
- Patel et al. (2022) - Impute and normalize before split
- Nnadi et al. (2026) - Grid search with 5-fold cross-validation
- Zhu et al. (2023) - SMOTE-ENN combined sampling method
- Duan et al. (2020) - NGBoost: Natural Gradient Boosting
- Mothilal et al. (2020) - DiCE: Diverse Counterfactual Explanations

In [ ]:
!pip install ngboost xgboost scikit-learn pandas numpy matplotlib seaborn dice-ml imbalanced-learn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, log_loss,
                             roc_curve, auc, roc_auc_score)
from sklearn.impute import SimpleImputer
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor
from sklearn.multiclass import OneVsRestClassifier

from ngboost import NGBClassifier
from ngboost.distns import k_categorical
from xgboost import XGBClassifier

from imblearn.combine import SMOTEENN

from scipy.stats import chi2

warnings.filterwarnings("ignore")

# Create figures directory
os.makedirs("figures", exist_ok=True)

print("All imports successful!")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PATH = "/content/drive/MyDrive/Canada_dataset.csv"
LOCAL_PATH = "Canada_dataset.csv"

if os.path.exists(DRIVE_PATH):
    df = pd.read_csv(DRIVE_PATH)
    print(f"Loaded from Drive: {DRIVE_PATH}")
elif os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f"Loaded locally: {LOCAL_PATH}")
else:
    from google.colab import files
    print("Upload Canada_dataset.csv:")
    uploaded = files.upload()
    import io
    df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))

print(f"Shape: {df.shape}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nDistribusi Kelas (CCME_WQI):\n{df['CCME_WQI'].value_counts()}")
print(f"\nPersentase Kelas:")
print(df["CCME_WQI"].value_counts(normalize=True).map("{:.1%}".format))

### Exploratory Data Analysis

Langkah ini mengikuti standar eksplorasi data sebelum preprocessing untuk memahami distribusi fitur, missing values, dan korelasi antar variabel. Referensi: Patel et al. (2022) melakukan EDA serupa untuk memahami karakteristik dataset sebelum pemodelan.

In [ ]:
# =============================================================================
# EXPLORATORY DATA ANALYSIS
# =============================================================================
print("="*70)
print("EXPLORATORY DATA ANALYSIS")
print("="*70)

# Define numeric features for Canada dataset
numeric_features = ['Ammonia (mg/l)', 'Biochemical Oxygen Demand (mg/l)',
                    'Dissolved Oxygen (mg/l)', 'Orthophosphate (mg/l)',
                    'pH (ph units)', 'Temperature (cel)',
                    'Nitrogen (mg/l)', 'Nitrate (mg/l)']

# Statistik Deskriptif
print("\nStatistik Deskriptif:")
display(df[numeric_features + ['CCME_WQI']].describe())

# Plot 1: Distribusi Kelas + Missing Values + Korelasi
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart kelas
class_counts = df["CCME_WQI"].value_counts()
colors_class = ["#2ecc71", "#3498db", "#f39c12", "#e67e22", "#e74c3c"]
bars = axes[0].bar(range(len(class_counts)), class_counts.values,
                   color=colors_class[:len(class_counts)])
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels(class_counts.index, rotation=45, ha='right')
axes[0].set_title("Distribusi Kelas CCME WQI", fontsize=13)
axes[0].set_ylabel("Jumlah Sampel")
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha="center", fontsize=10, fontweight="bold")

# Missing values
missing = df[numeric_features].isnull().sum()
axes[1].barh(range(len(numeric_features)), missing.values, color="#f39c12")
axes[1].set_yticks(range(len(numeric_features)))
axes[1].set_yticklabels([f.split(' (')[0] for f in numeric_features], fontsize=9)
axes[1].set_title("Missing Values per Feature", fontsize=13)
axes[1].set_xlabel("Jumlah Missing")
for i, val in enumerate(missing.values):
    axes[1].text(val + 0.5, i, str(val), va="center", fontsize=11)

# Heatmap korelasi
corr = df[numeric_features].corr()
mask_corr = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask_corr, annot=True, fmt=".2f", cmap="coolwarm",
            ax=axes[2], vmin=-1, vmax=1, center=0, square=True,
            annot_kws={"size": 7})
axes[2].set_title("Korelasi Antar Fitur", fontsize=13)

plt.tight_layout()
plt.savefig("figures/eda_canada_v2.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 2: Feature distributions
fig, axes2 = plt.subplots(2, 4, figsize=(16, 8))
axes2_flat = axes2.flatten()

for i, col in enumerate(numeric_features):
    axes2_flat[i].hist(df[col].dropna(), bins=30, alpha=0.7,
                       color='steelblue', edgecolor='white')
    axes2_flat[i].set_title(col.split(' (')[0], fontsize=10)

plt.suptitle('Distribusi Fitur - Canada Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/feature_distributions_canada_v2.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA complete.")

### Preprocessing - Impute, Scale, lalu Split

Langkah ini mengikuti Al Bataineh et al. (2026) Algorithm 3: "Step 1: 1.1 Handle missing values using median imputation, 1.2 Normalize all features to range [0,1], 1.3 Split D into training set D_train and testing set D_test." Meskipun dataset Canada tidak memiliki missing values, langkah imputation tetap dipertahankan untuk konsistensi pipeline. Pendekatan ini juga diterapkan oleh Patel et al. (2022).

In [ ]:
# =============================================================================
# PREPROCESSING - Al Bataineh et al. (2026) Algorithm 3
# Alur: Impute SELURUH data -> Scale SELURUH data -> BARU Split 70/15/15
# =============================================================================

# Features dan Label
feature_names = ['Ammonia (mg/l)', 'Biochemical Oxygen Demand (mg/l)',
                 'Dissolved Oxygen (mg/l)', 'Orthophosphate (mg/l)',
                 'pH (ph units)', 'Temperature (cel)',
                 'Nitrogen (mg/l)', 'Nitrate (mg/l)']

X = df[feature_names].copy()
y_raw = df["CCME_WQI"].copy()

# Encode target: 5 classes
le = LabelEncoder()
y = pd.Series(le.fit_transform(y_raw), name="CCME_WQI")
class_names = list(le.classes_)
n_classes = len(class_names)

print(f"Features: {feature_names}")
print(f"Label: CCME_WQI ({n_classes} classes: {class_names})")
print(f"Encoded mapping: {dict(zip(class_names, range(n_classes)))}")
print(f"Total samples: {len(df)}")
print(f"\nMissing values sebelum imputation:")
print(X.isnull().sum())

# STEP 1: Median Imputation pada SELURUH dataset (Al Bataineh et al., 2026 Step 1.1)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_names)
print(f"\nSetelah imputation - Missing values: {X_imputed.isnull().sum().sum()}")
print("(Dataset Canada tidak memiliki missing values, imputation untuk konsistensi pipeline)")

# STEP 2: MinMax Scaling pada SELURUH dataset (Al Bataineh et al., 2026 Step 1.2)
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_names)
print(f"Setelah scaling - Range: [{X_scaled.min().min():.4f}, {X_scaled.max().max():.4f}]")

# STEP 3: Split 70/15/15 stratified (Al Bataineh et al., 2026 Step 1.3)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.15, stratify=y, random_state=42)
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, stratify=y_temp, random_state=42)

# Convert to numpy arrays for model training
X_train_s = X_train.values
X_val_s = X_val.values
X_test_s = X_test.values

print(f"\nSplit Results:")
print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print("\nPreprocessing complete (Al Bataineh et al., 2026 Algorithm 3).")
print("Pipeline: Impute (full data) -> Scale (full data) -> Split 70/15/15")

### SMOTE-ENN Kondisional

Langkah ini mengikuti Zhu et al. (2023) yang menerapkan "SMOTE-ENN combined sampling method" untuk mengatasi ketidakseimbangan kelas. Kami membandingkan performa dengan dan tanpa SMOTE-ENN pada data latih multi-class, dan memilih pendekatan yang memberikan hasil terbaik.

In [ ]:
# =============================================================================
# SMOTE-ENN KONDISIONAL
# Referensi: Zhu et al. (2023) - SMOTE-ENN combined sampling
# =============================================================================

print("="*70)
print("SMOTE-ENN KONDISIONAL - Perbandingan dengan/tanpa resampling")
print("="*70)

# Check class distribution in training set
print(f"\nDistribusi kelas training set SEBELUM SMOTE-ENN:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Kelas {u} ({class_names[u]}): {c} ({c/len(y_train)*100:.1f}%)")

# Apply SMOTE-ENN on training data only
smote_enn = SMOTEENN(random_state=42)
X_train_resampled, y_train_resampled = smote_enn.fit_resample(X_train_s, y_train.values)

print(f"\nDistribusi kelas training set SETELAH SMOTE-ENN:")
unique_r, counts_r = np.unique(y_train_resampled, return_counts=True)
for u, c in zip(unique_r, counts_r):
    print(f"  Kelas {u} ({class_names[u]}): {c} ({c/len(y_train_resampled)*100:.1f}%)")
print(f"  Total: {len(y_train_resampled)} (dari {len(y_train)})")

# Train NGBoost WITH SMOTE-ENN
print("\nTraining NGBoost DENGAN SMOTE-ENN...")
try:
    ngb_smote = NGBClassifier(
        Dist=k_categorical(n_classes), n_estimators=300, learning_rate=0.05,
        minibatch_frac=0.8, col_sample=0.8, random_state=42, verbose=False
    )
    ngb_smote.fit(X_train_resampled, y_train_resampled,
                  X_val=X_val_s, Y_val=y_val.values)
    y_pred_smote = ngb_smote.predict(X_val_s)
    f1_smote = f1_score(y_val, y_pred_smote, average='macro')
    acc_smote = accuracy_score(y_val, y_pred_smote)
    smote_success = True
except Exception as e:
    print(f"  SMOTE-ENN training failed: {e}")
    f1_smote = 0.0
    acc_smote = 0.0
    smote_success = False

# Train NGBoost WITHOUT SMOTE-ENN
print("Training NGBoost TANPA SMOTE-ENN...")
ngb_no_smote = NGBClassifier(
    Dist=k_categorical(n_classes), n_estimators=300, learning_rate=0.05,
    minibatch_frac=0.8, col_sample=0.8, random_state=42, verbose=False
)
ngb_no_smote.fit(X_train_s, y_train.values, X_val=X_val_s, Y_val=y_val.values)
y_pred_no_smote = ngb_no_smote.predict(X_val_s)
f1_no_smote = f1_score(y_val, y_pred_no_smote, average='macro')
acc_no_smote = accuracy_score(y_val, y_pred_no_smote)

print(f"\n{'='*55}")
print(f"PERBANDINGAN (Validation Set - Macro F1):")
print(f"{'='*55}")
print(f"{'Metrik':<15} {'Dengan SMOTE-ENN':<20} {'Tanpa SMOTE-ENN':<20}")
print(f"{'-'*55}")
print(f"{'F1 (macro)':<15} {f1_smote:<20.4f} {f1_no_smote:<20.4f}")
print(f"{'Accuracy':<15} {acc_smote:<20.4f} {acc_no_smote:<20.4f}")

# Decision: use SMOTE-ENN only if it improves macro F1
use_smote = smote_success and (f1_smote > f1_no_smote)
if use_smote:
    X_train_final = X_train_resampled
    y_train_final = y_train_resampled
    print(f"\nKESIMPULAN: SMOTE-ENN MENINGKATKAN performa (F1 macro: {f1_no_smote:.4f} -> {f1_smote:.4f})")
    print("=> Menggunakan data dengan SMOTE-ENN untuk training selanjutnya.")
else:
    X_train_final = X_train_s
    y_train_final = y_train.values
    print(f"\nKESIMPULAN: SMOTE-ENN TIDAK meningkatkan performa (F1 macro: {f1_no_smote:.4f} vs {f1_smote:.4f})")
    print("=> Menggunakan data ASLI (tanpa SMOTE-ENN) untuk training selanjutnya.")
    print("   Temuan: Dataset Canada memiliki distribusi multi-class yang relatif seimbang,")
    print("   sehingga SMOTE-ENN tidak memberikan perbaikan signifikan.")

### Training NGBoost (Parameter Tetap - Diagram Metodologi)

Langkah ini mengikuti Duan et al. (2020) - "Natural Gradient Boosting for Probabilistic Prediction" dengan parameter FIXED sesuai diagram metodologi penelitian. Training ini dilakukan SEBELUM hyperparameter tuning untuk mendapatkan baseline performance NGBoost.

**Parameter Tetap (Diagram Metodologi):**
- Distribution: k_categorical(5) (5 kelas CCME WQI)
- n_estimators: 300
- learning_rate: 0.05
- minibatch_frac: 0.8
- col_sample: 0.8
- Base Learner: DecisionTreeRegressor(max_depth=4)
- Early Stopping: pada validation set

In [ ]:
# =============================================================================
# TRAINING NGBoost - Parameter Tetap (Diagram Metodologi)
# Referensi: Duan et al. (2020) - NGBoost
# =============================================================================

print("="*70)
print("TRAINING NGBoost (Parameter Tetap - Sebelum Tuning)")
print("="*70)

# NGBoost dengan parameter tetap dari diagram metodologi
ngb_initial = NGBClassifier(
    Dist=k_categorical(n_classes),
    Base=DecisionTreeRegressor(max_depth=4),
    n_estimators=300,
    learning_rate=0.05,
    minibatch_frac=0.8,
    col_sample=0.8,
    random_state=42,
    verbose=False
)

# Training dengan early stopping pada validation set
ngb_initial.fit(
    X_train_final, y_train_final,
    X_val=X_val_s, Y_val=y_val.values
)

# Evaluasi pada test set
y_pred_ngb_init = ngb_initial.predict(X_test_s)
y_prob_ngb_init = ngb_initial.predict_proba(X_test_s)

print(f"\nParameter:")
print(f"  Dist: k_categorical({n_classes})")
print(f"  n_estimators: 300")
print(f"  learning_rate: 0.05")
print(f"  minibatch_frac: 0.8")
print(f"  col_sample: 0.8")
print(f"  Base: DecisionTreeRegressor(max_depth=4)")
print(f"  Early stopping: Yes (validation set)")

print(f"\nHasil pada Test Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_ngb_init):.4f}")
print(f"  F1 (macro):  {f1_score(y_test, y_pred_ngb_init, average='macro'):.4f}")
print(f"  F1 (weighted): {f1_score(y_test, y_pred_ngb_init, average='weighted'):.4f}")
print(f"  Log Loss:  {log_loss(y_test, y_prob_ngb_init):.4f}")

print("\nNGBoost initial training complete!")

### Training Baseline Models (Parameter Default)

Langkah ini melatih model baseline XGBoost dan Random Forest dengan parameter default/reasonable SEBELUM hyperparameter tuning. Tujuannya adalah mendapatkan baseline performance untuk perbandingan dengan hasil setelah tuning.

**Referensi:**
- Chen & Guestrin (2016) - XGBoost: A Scalable Tree Boosting System
- Breiman (2001) - Random Forests

In [ ]:
# =============================================================================
# TRAINING BASELINE MODELS - Parameter Default
# XGBoost dan Random Forest sebelum tuning
# =============================================================================

print("="*70)
print("TRAINING BASELINE MODELS (Parameter Default - Sebelum Tuning)")
print("="*70)

# XGBoost dengan parameter default/reasonable (multi-class)
print("\n[1/2] Training XGBoost (default params, multi:softprob)...")
xgb_initial = XGBClassifier(
    objective='multi:softprob',
    num_class=n_classes,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.3,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
xgb_initial.fit(X_train_final, y_train_final)

y_pred_xgb_init = xgb_initial.predict(X_test_s)
y_prob_xgb_init = xgb_initial.predict_proba(X_test_s)

print(f"  Accuracy:  {accuracy_score(y_test, y_pred_xgb_init):.4f}")
print(f"  F1 (macro):  {f1_score(y_test, y_pred_xgb_init, average='macro'):.4f}")

# Random Forest dengan parameter default
print("\n[2/2] Training Random Forest (default params)...")
rf_initial = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf_initial.fit(X_train_final, y_train_final)

y_pred_rf_init = rf_initial.predict(X_test_s)
y_prob_rf_init = rf_initial.predict_proba(X_test_s)

print(f"  Accuracy:  {accuracy_score(y_test, y_pred_rf_init):.4f}")
print(f"  F1 (macro):  {f1_score(y_test, y_pred_rf_init, average='macro'):.4f}")

# Store initial results for comparison
initial_models = {
    'NGBoost': ngb_initial,
    'XGBoost': xgb_initial,
    'Random Forest': rf_initial
}
results_initial = {}
for name, model in initial_models.items():
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)
    results_initial[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (macro)': precision_score(y_test, y_pred, average='macro'),
        'Precision (weighted)': precision_score(y_test, y_pred, average='weighted'),
        'Recall (macro)': recall_score(y_test, y_pred, average='macro'),
        'Recall (weighted)': recall_score(y_test, y_pred, average='weighted'),
        'F1 (macro)': f1_score(y_test, y_pred, average='macro'),
        'F1 (weighted)': f1_score(y_test, y_pred, average='weighted'),
        'Log Loss': log_loss(y_test, y_prob),
    }

print("\n" + "="*70)
print("RINGKASAN BASELINE (Sebelum Tuning):")
print("="*70)
print(f"\n{'Model':<20} {'Accuracy':<12} {'F1 (macro)':<12} {'Log Loss':<12}")
print("-"*56)
for name in results_initial:
    r = results_initial[name]
    print(f"{name:<20} {r['Accuracy']:<12.4f} {r['F1 (macro)']:<12.4f} "
          f"{r['Log Loss']:<12.4f}")
print("\nBaseline training complete! Lanjut ke Hyperparameter Tuning...")

### Hyperparameter Tuning (Grid Search + 5-Fold CV)

Langkah ini mengikuti Nnadi et al. (2026) yang menyatakan "Model hyperparameters were optimized via grid search with five-fold cross-validation on the training data." Parameter grid untuk NGBoost mengacu pada Zhu et al. (2023) dan Duan et al. (2020) yang menggunakan "decision tree base learner max_depth=3 (default), learning rate 0.01" serta Zhu et al. (2023) "decision tree criterion friedman_mse, max_depth=8, n_estimators=224, learning_rate=0.0237".

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING - Grid Search + 5-Fold CV
# Referensi: Nnadi et al. (2026), Zhu et al. (2023), Duan et al. (2020)
# =============================================================================

print("="*70)
print("HYPERPARAMETER TUNING (Grid Search + 5-Fold Cross-Validation)")
print("="*70)

# --- NGBoost Grid Search (manual implementation) ---
print("\n[1/3] NGBoost Grid Search (k_categorical(5))...")
print("Parameter grid:")
ngb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'minibatch_frac': [0.5, 0.8, 1.0],
    'max_depth': [3, 4, 5, 8]
}
print(f"  n_estimators: {ngb_param_grid['n_estimators']}")
print(f"  learning_rate: {ngb_param_grid['learning_rate']}")
print(f"  minibatch_frac: {ngb_param_grid['minibatch_frac']}")
print(f"  max_depth (base learner): {ngb_param_grid['max_depth']}")

from itertools import product
import random

random.seed(42)
all_combos = list(product(
    ngb_param_grid['n_estimators'],
    ngb_param_grid['learning_rate'],
    ngb_param_grid['minibatch_frac'],
    ngb_param_grid['max_depth']
))

# Sample 20 combinations for efficiency (full grid = 192 combos)
n_iter = min(20, len(all_combos))
sampled_combos = random.sample(all_combos, n_iter)
print(f"  Sampling {n_iter} dari {len(all_combos)} kombinasi total")

best_ngb_score = -1
best_ngb_params = {}

skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for idx, (n_est, lr, mb_frac, md_val) in enumerate(sampled_combos):
    scores = []
    for train_idx, val_idx in skf_tune.split(X_train_final, y_train_final):
        X_cv_train = X_train_final[train_idx]
        X_cv_val = X_train_final[val_idx]
        y_cv_train = y_train_final[train_idx]
        y_cv_val = y_train_final[val_idx]

        ngb_cv = NGBClassifier(
            Dist=k_categorical(n_classes),
            Base=DecisionTreeRegressor(max_depth=md_val),
            n_estimators=n_est,
            learning_rate=lr,
            minibatch_frac=mb_frac,
            col_sample=0.8,
            random_state=42,
            verbose=False
        )
        ngb_cv.fit(X_cv_train, y_cv_train, X_val=X_cv_val, Y_val=y_cv_val)
        y_cv_pred = ngb_cv.predict(X_cv_val)
        scores.append(f1_score(y_cv_val, y_cv_pred, average='macro'))

    mean_score = np.mean(scores)
    if mean_score > best_ngb_score:
        best_ngb_score = mean_score
        best_ngb_params = {
            'n_estimators': n_est, 'learning_rate': lr,
            'minibatch_frac': mb_frac, 'max_depth': md_val
        }

    if (idx + 1) % 5 == 0:
        print(f"  Progress: {idx+1}/{n_iter} combinations evaluated")

print(f"\n  Best NGBoost params: {best_ngb_params}")
print(f"  Best NGBoost CV F1 (macro): {best_ngb_score:.4f}")

# --- XGBoost Grid Search ---
print("\n[2/3] XGBoost Grid Search (sklearn GridSearchCV)...")
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9]
}
print(f"  Total combinations: {4*5*3*3}")

xgb_base = XGBClassifier(
    objective='multi:softprob', num_class=n_classes,
    random_state=42, eval_metric='mlogloss', verbosity=0,
    colsample_bytree=0.9
)
xgb_grid = GridSearchCV(
    xgb_base, xgb_param_grid, cv=5, scoring='f1_macro',
    n_jobs=-1, verbose=0
)
xgb_grid.fit(X_train_final, y_train_final)
best_xgb_params = xgb_grid.best_params_
print(f"  Best XGBoost params: {best_xgb_params}")
print(f"  Best XGBoost CV F1 (macro): {xgb_grid.best_score_:.4f}")

# --- Random Forest Grid Search ---
print("\n[3/3] Random Forest Grid Search (sklearn GridSearchCV)...")
rf_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 15, 20],
    'min_samples_split': [2, 5, 10]
}
print(f"  Total combinations: {4*4*3}")

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid = GridSearchCV(
    rf_base, rf_param_grid, cv=5, scoring='f1_macro',
    n_jobs=-1, verbose=0
)
rf_grid.fit(X_train_final, y_train_final)
best_rf_params = rf_grid.best_params_
print(f"  Best Random Forest params: {best_rf_params}")
print(f"  Best RF CV F1 (macro): {rf_grid.best_score_:.4f}")

print("\n" + "="*70)
print("HYPERPARAMETER TUNING SELESAI")
print("="*70)

### Training Models dengan Best Hyperparameters

Langkah ini mengikuti Duan et al. (2020) untuk konfigurasi NGBoost: "Natural Gradient Boosting for Probabilistic Prediction" dengan k_categorical(5) distribution untuk klasifikasi multi-class (5 kelas CCME WQI), menggunakan Fisher Information Matrix sebagai natural gradient, dan early stopping pada validation set.

In [ ]:
# =============================================================================
# TRAINING MODELS - Dengan Best Hyperparameters
# NGBoost (k_categorical(5)) + XGBoost + Random Forest
# Referensi: Duan et al. (2020)
# =============================================================================

# NGBoost with best parameters
print("Training NGBoost with best hyperparameters (k_categorical(5))...")
ngb = NGBClassifier(
    Dist=k_categorical(n_classes),
    Base=DecisionTreeRegressor(max_depth=best_ngb_params['max_depth']),
    n_estimators=best_ngb_params['n_estimators'],
    learning_rate=best_ngb_params['learning_rate'],
    minibatch_frac=best_ngb_params['minibatch_frac'],
    col_sample=0.8,
    random_state=42,
    verbose=False
)
ngb.fit(X_train_final, y_train_final, X_val=X_val_s, Y_val=y_val.values)
print(f"NGBoost done! (n_estimators={best_ngb_params['n_estimators']}, "
      f"lr={best_ngb_params['learning_rate']}, "
      f"max_depth={best_ngb_params['max_depth']})")

# XGBoost with best parameters
print("\nTraining XGBoost with best hyperparameters...")
xgb = XGBClassifier(
    objective='multi:softprob', num_class=n_classes,
    n_estimators=best_xgb_params['n_estimators'],
    max_depth=best_xgb_params['max_depth'],
    learning_rate=best_xgb_params['learning_rate'],
    subsample=best_xgb_params['subsample'],
    colsample_bytree=0.9,
    random_state=42, eval_metric='mlogloss', verbosity=0
)
xgb.fit(X_train_final, y_train_final)
print(f"XGBoost done! (params: {best_xgb_params})")

# Random Forest with best parameters
print("\nTraining Random Forest with best hyperparameters...")
rf = RandomForestClassifier(
    n_estimators=best_rf_params['n_estimators'],
    max_depth=best_rf_params['max_depth'],
    min_samples_split=best_rf_params['min_samples_split'],
    random_state=42, n_jobs=-1
)
rf.fit(X_train_final, y_train_final)
print(f"Random Forest done! (params: {best_rf_params})")

print("\n" + "="*50)
print("All models trained with optimized hyperparameters!")
print("="*50)

### Evaluasi Model - Perbandingan Sebelum vs Sesudah Tuning

Evaluasi menggunakan metrik standar klasifikasi multi-class: Accuracy, Precision (macro/weighted), Recall (macro/weighted), F1-Score (macro/weighted), Log Loss. Perbandingan dilakukan antara model INITIAL (parameter tetap/default) dengan model TUNED (setelah hyperparameter optimization).

**Referensi:**
- Guo et al. (2017) - On Calibration of Modern Neural Networks (ECE)
- Nnadi et al. (2026) - Evaluasi performa setelah hyperparameter tuning

In [ ]:
# =============================================================================
# EVALUASI MODEL - Perbandingan SEBELUM vs SESUDAH Tuning
# =============================================================================

# Tuned model predictions
tuned_models = {'NGBoost': ngb, 'XGBoost': xgb, 'Random Forest': rf}
results = {}  # results for tuned models (used by visualizations)
results_tuned = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)

    acc = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro')
    prec_weighted = precision_score(y_test, y_pred, average='weighted')
    rec_macro = recall_score(y_test, y_pred, average='macro')
    rec_weighted = recall_score(y_test, y_pred, average='weighted')
    f1_macro_val = f1_score(y_test, y_pred, average='macro')
    f1_weighted_val = f1_score(y_test, y_pred, average='weighted')
    nll = log_loss(y_test, y_prob)

    results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'Accuracy': acc,
        'Precision (macro)': prec_macro,
        'Precision (weighted)': prec_weighted,
        'Recall (macro)': rec_macro,
        'Recall (weighted)': rec_weighted,
        'F1 (macro)': f1_macro_val,
        'F1 (weighted)': f1_weighted_val,
        'Log Loss': nll
    }
    results_tuned[name] = results[name]

# --- Comparison Table ---
print("="*70)
print("EVALUASI MODEL: SEBELUM vs SESUDAH Hyperparameter Tuning")
print("Pipeline: Al Bataineh et al. (2026) + SMOTE-ENN + Grid Search")
print("="*70)

metrics_list = ['Accuracy', 'Precision (macro)', 'Precision (weighted)',
                'Recall (macro)', 'Recall (weighted)',
                'F1 (macro)', 'F1 (weighted)', 'Log Loss']

for name in ['NGBoost', 'XGBoost', 'Random Forest']:
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  {'Metric':<22} {'Initial':<12} {'Tuned':<12} {'Delta':<12}")
    print(f"  {'-'*58}")
    for m in metrics_list:
        val_i = results_initial[name][m]
        val_t = results_tuned[name][m]
        delta = val_t - val_i
        arrow = '+' if delta > 0 else ''
        print(f"  {m:<22} {val_i:<12.4f} {val_t:<12.4f} {arrow}{delta:<12.4f}")

# Overall summary
print("\n" + "="*70)
print("RINGKASAN PERBANDINGAN (Test Set):")
print("="*70)
print(f"\n{'Model':<20} {'F1(m) Initial':<16} {'F1(m) Tuned':<16} {'Improvement':<14}")
print("-"*66)
for name in ['NGBoost', 'XGBoost', 'Random Forest']:
    f1_i = results_initial[name]['F1 (macro)']
    f1_t = results_tuned[name]['F1 (macro)']
    imp = (f1_t - f1_i) / f1_i * 100 if f1_i > 0 else 0
    print(f"{name:<20} {f1_i:<16.4f} {f1_t:<16.4f} {imp:+.2f}%")

print("\n" + "="*70)
print("Classification Reports (Tuned Models):")
print("="*70)
for name in tuned_models:
    print(f"\n--- {name} (Tuned) ---")
    print(classification_report(y_test, results[name]['y_pred'],
                                target_names=class_names))

In [ ]:
# =============================================================================
# McNEMAR'S TEST
# Referensi: McNemar (1947) - perbandingan statistik antar classifier
# =============================================================================

def mcnemar_test(y_true, y_pred1, y_pred2, name1, name2):
    """Perform McNemar test between two classifiers."""
    correct1 = (y_pred1 == y_true)
    correct2 = (y_pred2 == y_true)

    # Contingency table
    b = np.sum(correct1 & ~correct2)  # model1 correct, model2 wrong
    c = np.sum(~correct1 & correct2)  # model1 wrong, model2 correct

    # McNemar statistic with continuity correction
    if (b + c) == 0:
        statistic = 0.0
        p_value = 1.0
    else:
        statistic = (abs(b - c) - 1)**2 / (b + c)
        p_value = 1 - chi2.cdf(statistic, df=1)

    return statistic, p_value, b, c

print("="*70)
print("McNEMAR'S TEST - Perbandingan Statistik antar Model")
print("="*70)
print("H0: Kedua model memiliki performa yang sama")
print("H1: Kedua model memiliki performa yang berbeda")
print(f"Significance level: alpha = 0.05")
print()

pairs = [
    ('NGBoost', 'XGBoost'),
    ('NGBoost', 'Random Forest'),
    ('XGBoost', 'Random Forest')
]

for name1, name2 in pairs:
    stat, pval, b, c = mcnemar_test(
        y_test.values,
        results[name1]['y_pred'],
        results[name2]['y_pred'],
        name1, name2
    )
    sig = "SIGNIFICANT" if pval < 0.05 else "NOT significant"
    print(f"{name1} vs {name2}:")
    print(f"  b={b} (only {name1} correct), c={c} (only {name2} correct)")
    print(f"  Chi-squared = {stat:.4f}, p-value = {pval:.4f} -> {sig}")
    print()

In [ ]:
# =============================================================================
# VISUALISASI: CONFUSION MATRICES
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, ['NGBoost', 'XGBoost', 'Random Forest']):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 10})
    ax.set_title(f'{name}\nAcc={results[name]["Accuracy"]:.4f}', fontsize=12)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Confusion Matrices - Canada Dataset (V2 Pipeline)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/confusion_matrices_canada_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/confusion_matrices_canada_v2.png")

In [ ]:
# =============================================================================
# VISUALISASI: ROC CURVES (One-vs-Rest Multi-class)
# =============================================================================

# Binarize the test labels for OvR ROC
y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_roc = plt.cm.Set1(np.linspace(0, 1, n_classes))

for ax, name in zip(axes, ['NGBoost', 'XGBoost', 'Random Forest']):
    y_prob = results[name]['y_prob']
    macro_auc = 0.0

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc_i = auc(fpr, tpr)
        macro_auc += roc_auc_i
        ax.plot(fpr, tpr, color=colors_roc[i], linewidth=1.5,
                label=f'{class_names[i]} (AUC={roc_auc_i:.3f})')

    macro_auc /= n_classes
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'{name}\nMacro AUC={macro_auc:.4f}', fontsize=12)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.suptitle('ROC Curves (One-vs-Rest) - Canada Dataset (V2 Pipeline)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/roc_curves_canada_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/roc_curves_canada_v2.png")

In [ ]:
# =============================================================================
# VISUALISASI: FEATURE IMPORTANCE
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# NGBoost: Permutation Importance
print("Calculating NGBoost permutation importance...")
perm_imp = permutation_importance(ngb, X_test_s, y_test, n_repeats=10, random_state=42)
ngb_imp = perm_imp.importances_mean
sorted_idx = np.argsort(ngb_imp)
short_names = [f.split(' (')[0] for f in feature_names]
axes[0].barh(range(len(feature_names)), ngb_imp[sorted_idx], color='#e74c3c')
axes[0].set_yticks(range(len(feature_names)))
axes[0].set_yticklabels([short_names[i] for i in sorted_idx], fontsize=9)
axes[0].set_title('NGBoost\n(Permutation Importance)', fontsize=12)
axes[0].set_xlabel('Importance')

# XGBoost: Built-in importance
xgb_imp = xgb.feature_importances_
sorted_idx = np.argsort(xgb_imp)
axes[1].barh(range(len(feature_names)), xgb_imp[sorted_idx], color='#3498db')
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels([short_names[i] for i in sorted_idx], fontsize=9)
axes[1].set_title('XGBoost\n(Built-in Importance)', fontsize=12)
axes[1].set_xlabel('Importance')

# Random Forest: Built-in importance
rf_imp = rf.feature_importances_
sorted_idx = np.argsort(rf_imp)
axes[2].barh(range(len(feature_names)), rf_imp[sorted_idx], color='#2ecc71')
axes[2].set_yticks(range(len(feature_names)))
axes[2].set_yticklabels([short_names[i] for i in sorted_idx], fontsize=9)
axes[2].set_title('Random Forest\n(Built-in Importance)', fontsize=12)
axes[2].set_xlabel('Importance')

plt.suptitle('Feature Importance Comparison - Canada Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/feature_importance_canada_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/feature_importance_canada_v2.png")

### 5-Fold Cross-Validation

Langkah ini mengikuti Nnadi et al. (2026) untuk validasi robustness model. Preprocessing (imputation + scaling) dilakukan pada seluruh data sebelum CV split, konsisten dengan metodologi Al Bataineh et al. (2026). Pada setiap fold, model dilatih dan dievaluasi untuk mendapatkan estimasi performa yang stabil.

In [ ]:
# =============================================================================
# 5-FOLD CROSS-VALIDATION
# Preprocessing: Impute+Scale pada full data, lalu CV split
# Referensi: Al Bataineh et al. (2026), Nnadi et al. (2026)
# =============================================================================

print("="*70)
print("5-FOLD CROSS-VALIDATION")
print("Preprocessing: Impute -> Scale -> CV Split (Al Bataineh et al., 2026)")
print("="*70)

# Data sudah di-impute dan di-scale (X_scaled)
X_cv_full = X_scaled.values
y_cv_full = y.values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {'NGBoost': [], 'XGBoost': [], 'Random Forest': []}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_cv_full, y_cv_full), 1):
    X_tr, X_te = X_cv_full[train_idx], X_cv_full[test_idx]
    y_tr, y_te = y_cv_full[train_idx], y_cv_full[test_idx]

    # NGBoost
    ngb_fold = NGBClassifier(
        Dist=k_categorical(n_classes),
        Base=DecisionTreeRegressor(max_depth=best_ngb_params['max_depth']),
        n_estimators=best_ngb_params['n_estimators'],
        learning_rate=best_ngb_params['learning_rate'],
        minibatch_frac=best_ngb_params['minibatch_frac'],
        col_sample=0.8, random_state=42, verbose=False
    )
    ngb_fold.fit(X_tr, y_tr)
    cv_results['NGBoost'].append(f1_score(y_te, ngb_fold.predict(X_te), average='macro'))

    # XGBoost
    xgb_fold = XGBClassifier(
        objective='multi:softprob', num_class=n_classes,
        n_estimators=best_xgb_params['n_estimators'],
        max_depth=best_xgb_params['max_depth'],
        learning_rate=best_xgb_params['learning_rate'],
        subsample=best_xgb_params['subsample'],
        colsample_bytree=0.9, random_state=42, eval_metric='mlogloss', verbosity=0
    )
    xgb_fold.fit(X_tr, y_tr)
    cv_results['XGBoost'].append(f1_score(y_te, xgb_fold.predict(X_te), average='macro'))

    # Random Forest
    rf_fold = RandomForestClassifier(
        n_estimators=best_rf_params['n_estimators'],
        max_depth=best_rf_params['max_depth'],
        min_samples_split=best_rf_params['min_samples_split'],
        random_state=42, n_jobs=-1
    )
    rf_fold.fit(X_tr, y_tr)
    cv_results['Random Forest'].append(f1_score(y_te, rf_fold.predict(X_te), average='macro'))

    print(f"  Fold {fold}: NGBoost={cv_results['NGBoost'][-1]:.4f}, "
          f"XGBoost={cv_results['XGBoost'][-1]:.4f}, "
          f"RF={cv_results['Random Forest'][-1]:.4f}")

print(f"\n{'='*60}")
print(f"{'Model':<15} {'Mean F1 (macro)':<20} {'Std':<15}")
print(f"{'-'*50}")
for name in cv_results:
    scores = cv_results[name]
    print(f"{name:<15} {np.mean(scores):<20.4f} {np.std(scores):<15.4f}")
print(f"{'='*60}")

### Analisis Kalibrasi

Analisis kalibrasi model mengukur seberapa baik probabilitas prediksi mencerminkan probabilitas sebenarnya. Model yang well-calibrated memiliki kurva kalibrasi mendekati garis diagonal. Untuk multi-class, analisis dilakukan per kelas menggunakan pendekatan One-vs-Rest.

**Referensi:**
- Guo et al. (2017) - On Calibration of Modern Neural Networks
- Duan et al. (2020) - NGBoost probabilistic prediction advantages

In [ ]:
# =============================================================================
# ANALISIS KALIBRASI (Multi-class)
# Referensi: Guo et al. (2017) - On Calibration of Modern Neural Networks
# =============================================================================

print("="*70)
print("ANALISIS KALIBRASI MODEL")
print("="*70)

def calculate_ece_multiclass(y_true, y_prob, n_bins=10):
    """Calculate Expected Calibration Error for multi-class."""
    n_classes_local = y_prob.shape[1]
    ece_total = 0.0
    total_samples = 0
    for c in range(n_classes_local):
        y_binary = (y_true == c).astype(int)
        prob_c = y_prob[:, c]
        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        for i in range(n_bins):
            mask = (prob_c >= bin_boundaries[i]) & (prob_c < bin_boundaries[i+1])
            if mask.sum() > 0:
                bin_acc = y_binary[mask].mean()
                bin_conf = prob_c[mask].mean()
                ece_total += mask.sum() * abs(bin_acc - bin_conf)
                total_samples += mask.sum()
    return ece_total / total_samples if total_samples > 0 else 0.0

# Compute ECE for tuned models
for name in results:
    results[name]['ECE'] = calculate_ece_multiclass(y_test.values, results[name]['y_prob'])

# Also compute ECE for initial models
for name in results_initial:
    results_initial[name]['ECE'] = calculate_ece_multiclass(y_test.values, results_initial[name]['y_prob'])

# --- Calibration Curves (One-vs-Rest) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, ['NGBoost', 'XGBoost', 'Random Forest']):
    y_prob = results[name]['y_prob']
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect')

    for i in range(n_classes):
        prob_true, prob_pred = calibration_curve(
            (y_test.values == i).astype(int),
            y_prob[:, i], n_bins=10, strategy='uniform'
        )
        ax.plot(prob_pred, prob_true, 's-', linewidth=1.5, markersize=4,
                label=f'{class_names[i]}')

    ax.set_xlabel('Mean Predicted Probability', fontsize=10)
    ax.set_ylabel('Fraction of Positives', fontsize=10)
    ax.set_title(f'{name} (ECE={results[name]["ECE"]:.4f})', fontsize=12)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.suptitle('Calibration Curves (One-vs-Rest) - Canada Dataset (V2 Pipeline)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/calibration_canada_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/calibration_canada_v2.png")

# --- ECE Comparison ---
print("\n" + "="*70)
print("ECE COMPARISON (Initial vs Tuned):")
print("="*70)
print(f"\n{'Model':<20} {'ECE Initial':<15} {'ECE Tuned':<15} {'Improvement':<15}")
print("-"*65)
for name in ['NGBoost', 'XGBoost', 'Random Forest']:
    ece_i = results_initial[name]['ECE']
    ece_t = results[name]['ECE']
    delta = ece_i - ece_t  # lower ECE is better
    print(f"{name:<20} {ece_i:<15.4f} {ece_t:<15.4f} {delta:+.4f}")

# --- Uncertainty Zone Analysis (multi-class) ---
print("\n" + "="*70)
print("UNCERTAINTY ZONE ANALYSIS (based on max predicted probability)")
print("="*70)
print("\nZona berdasarkan max P(class):")
print("Zone 1: max_P >= 0.8 (Very Confident)")
print("Zone 2: 0.6 <= max_P < 0.8 (Confident)")
print("Zone 3: 0.4 <= max_P < 0.6 (Moderate)")
print("Zone 4: max_P < 0.4 (Uncertain)")

zones = [
    ("Zone 1: max_P >= 0.8 (Very Confident)", 0.8, 1.01),
    ("Zone 2: 0.6-0.8 (Confident)", 0.6, 0.8),
    ("Zone 3: 0.4-0.6 (Moderate)", 0.4, 0.6),
    ("Zone 4: max_P < 0.4 (Uncertain)", 0.0, 0.4)
]

for name in ['NGBoost', 'XGBoost', 'Random Forest']:
    print(f"\n{'='*50}")
    print(f"Model: {name} (Tuned)")
    print(f"{'='*50}")
    y_prob = results[name]['y_prob']
    max_probs = np.max(y_prob, axis=1)
    print(f"{'Zone':<35} {'N':<8} {'Accuracy':<12} {'Avg max_P':<15}")
    print("-"*70)
    for zone_name, low, high in zones:
        mask = (max_probs >= low) & (max_probs < high)
        n = mask.sum()
        if n > 0:
            zone_acc = accuracy_score(y_test.values[mask], results[name]['y_pred'][mask])
            avg_prob = max_probs[mask].mean()
            print(f"{zone_name:<35} {n:<8} {zone_acc:<12.4f} {avg_prob:<15.4f}")
        else:
            print(f"{zone_name:<35} {0:<8} {'N/A':<12} {'N/A':<15}")

print("\n" + "="*70)
print("INSIGHT: NGBoost memberikan informasi uncertainty yang memungkinkan:")
print("- Resource allocation: sampel uncertain -> pengujian tambahan")
print("- Risk management: keputusan berbeda untuk prediksi yakin vs tidak yakin")
print("- Transparency: end-user tahu kapan model 'ragu'")

### Prediksi Kualitas Air

Bagian ini mendemonstrasikan penggunaan model NGBoost yang telah dioptimasi untuk melakukan prediksi kualitas air multi-class. NGBoost memberikan distribusi probabilitas penuh di seluruh 5 kelas CCME WQI (Excellent, Good, Fair, Marginal, Poor), yang menunjukkan tingkat keyakinan model untuk setiap kemungkinan klasifikasi.

**Referensi:**
- Duan et al. (2020) - NGBoost: Natural Gradient Boosting for Probabilistic Prediction
- Keunggulan probabilistic prediction untuk decision making di water quality management

In [ ]:
# =============================================================================
# PREDIKSI KUALITAS AIR
# Demonstrasi prediksi dengan model NGBoost terbaik (tuned)
# Referensi: Duan et al. (2020) - Probabilistic Prediction
# =============================================================================

print("="*70)
print("PREDIKSI KUALITAS AIR - NGBoost (Tuned, Multi-class)")
print("="*70)

# Select representative samples from test set
y_prob_all = ngb.predict_proba(X_test_s)
y_pred_all = ngb.predict(X_test_s)
max_probs_all = np.max(y_prob_all, axis=1)

# Pick samples with varying confidence levels
demo_indices = []
target_confs = [0.95, 0.80, 0.60, 0.45, 0.35, 0.25]
for tc in target_confs:
    idx = np.argmin(np.abs(max_probs_all - tc))
    if idx not in demo_indices:
        demo_indices.append(idx)

print(f"\nDemonstrasi prediksi pada {len(demo_indices)} sampel representatif:")
print("\n" + "="*70)

for rank, idx in enumerate(demo_indices, 1):
    sample = X_test_s[idx:idx+1]
    pred = y_pred_all[idx]
    prob_all = y_prob_all[idx]
    true_label = y_test.values[idx]
    confidence = max_probs_all[idx] * 100

    pred_class = class_names[pred]
    true_class = class_names[true_label]
    correct = "BENAR" if pred == true_label else "SALAH"

    # Confidence category
    if confidence >= 80:
        conf_cat = "Sangat Yakin"
    elif confidence >= 60:
        conf_cat = "Cukup Yakin"
    else:
        conf_cat = "Tidak Yakin (Uncertainty Zone)"

    print(f"\n--- Sampel #{rank} ---")
    print(f"  Input Features (scaled):")
    for fi, fname in enumerate(feature_names):
        print(f"    {fname}: {sample[0, fi]:.4f}")

    print(f"\n  Distribusi Probabilitas (5 kelas CCME WQI):")
    for ci, cn in enumerate(class_names):
        bar = '|' * int(prob_all[ci] * 30)
        print(f"    P({cn:<10}) = {prob_all[ci]:.4f} {bar}")

    print(f"\n  Prediksi: {pred_class}")
    print(f"  Confidence: {confidence:.1f}% ({conf_cat})")
    print(f"  Label Sebenarnya: {true_class} -> Prediksi {correct}")

    # Decision recommendation
    if confidence < 60:
        print(f"  REKOMENDASI: Lakukan pengujian tambahan (confidence rendah)")
    elif pred in [class_names.index(c) for c in ['Marginal', 'Poor'] if c in class_names]:
        print(f"  REKOMENDASI: Kualitas air buruk, perlu treatment")
    else:
        print(f"  REKOMENDASI: Kualitas air memenuhi standar")

print("\n" + "="*70)
print("RINGKASAN PREDIKSI:")
print("="*70)
print(f"""
Keunggulan NGBoost untuk Prediksi Kualitas Air (Multi-class):
1. Memberikan DISTRIBUSI PROBABILITAS di seluruh 5 kelas CCME WQI
2. Tingkat keyakinan membantu pengambilan keputusan:
   - Confidence tinggi -> keputusan langsung
   - Confidence rendah -> perlu verifikasi pengujian
3. Multi-class output lebih informatif dari binary classification
4. Uncertainty quantification penting untuk pengelolaan kualitas air
""")

### Counterfactual Analysis with DiCE (Binary Grouping)

Langkah ini mengikuti Mothilal et al. (2020) untuk menghasilkan Diverse Counterfactual Explanations. Untuk dataset Canada multi-class, kami menggunakan **binary grouping**:
- {Marginal, Poor} = "Unacceptable"
- {Excellent, Good, Fair} = "Acceptable"

Target counterfactual: Unacceptable -> Acceptable

Pendekatan binary grouping memungkinkan penggunaan DiCE yang lebih interpretatif untuk rekomendasi perbaikan kualitas air. Referensi: Lenatti et al. (2025) untuk counterfactual multi-class dengan pengelompokan biner.

In [ ]:
# =============================================================================
# DiCE SETUP - Binary Grouping for Multi-class
# Referensi: Mothilal et al. (2020), Lenatti et al. (2025)
# =============================================================================
import dice_ml

print("="*70)
print("DiCE SETUP - Binary Grouping")
print("="*70)

# Binary grouping mapping
print(f"\nOriginal classes: {class_names}")
print(f"Encoded: {dict(zip(class_names, range(n_classes)))}")

# Map to binary: Acceptable = {Excellent, Good, Fair}, Unacceptable = {Marginal, Poor}
acceptable_classes = ['Excellent', 'Good', 'Fair']
unacceptable_classes = ['Marginal', 'Poor']

acceptable_encoded = [class_names.index(c) for c in acceptable_classes if c in class_names]
unacceptable_encoded = [class_names.index(c) for c in unacceptable_classes if c in class_names]

print(f"\nBinary Grouping:")
print(f"  Acceptable (encoded): {acceptable_encoded} = {[class_names[i] for i in acceptable_encoded]}")
print(f"  Unacceptable (encoded): {unacceptable_encoded} = {[class_names[i] for i in unacceptable_encoded]}")

def map_to_binary(pred):
    """Map 5-class prediction to binary (0=Acceptable, 1=Unacceptable)."""
    if isinstance(pred, (list, np.ndarray)):
        return np.array([1 if p in unacceptable_encoded else 0 for p in pred])
    return 1 if pred in unacceptable_encoded else 0

# NGBoost Multi-class Wrapper for DiCE (binary output)
class NGBoostBinaryWrapper:
    """Wraps NGBoost multi-class model to output binary predictions for DiCE.
    Internally: predicts 5 classes, then maps to binary.
    Data is ALREADY scaled (preprocessing done on full data before split).
    """
    def __init__(self, model, acceptable_enc, unacceptable_enc):
        self.model = model
        self.acceptable_enc = acceptable_enc
        self.unacceptable_enc = unacceptable_enc

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X_arr = X.values
        else:
            X_arr = np.array(X)
        preds_5class = self.model.predict(X_arr)
        return map_to_binary(preds_5class)

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X_arr = X.values
        else:
            X_arr = np.array(X)
        proba_5class = self.model.predict_proba(X_arr)
        # Sum probabilities: P(Acceptable) = sum of Excellent, Good, Fair
        p_acceptable = proba_5class[:, self.acceptable_enc].sum(axis=1)
        p_unacceptable = proba_5class[:, self.unacceptable_enc].sum(axis=1)
        return np.column_stack([p_acceptable, p_unacceptable])

ngb_binary_wrapper = NGBoostBinaryWrapper(ngb, acceptable_encoded, unacceptable_encoded)

# Test wrapper
y_pred_test_5class = ngb.predict(X_test_s)
y_binary_test = map_to_binary(y_pred_test_5class)
print(f"\nTest predictions (5-class): {np.unique(y_pred_test_5class, return_counts=True)}")
print(f"Test predictions (binary): Acceptable={np.sum(y_binary_test==0)}, Unacceptable={np.sum(y_binary_test==1)}")

# WHO/CCME constraints for permitted_range (in SCALED space [0,1])
permitted_range_original = {
    'Ammonia (mg/l)': [0, 2],
    'Biochemical Oxygen Demand (mg/l)': [0, 6],
    'Dissolved Oxygen (mg/l)': [4, 14],
    'Orthophosphate (mg/l)': [0, 1],
    'pH (ph units)': [6.5, 8.5],
    'Temperature (cel)': [0, 35],
    'Nitrogen (mg/l)': [0, 10],
    'Nitrate (mg/l)': [0, 50]
}

# Convert to scaled space
permitted_range = {}
for i, col in enumerate(feature_names):
    lo_orig, hi_orig = permitted_range_original[col]
    col_min = scaler.data_min_[i]
    col_max = scaler.data_max_[i]
    col_range = col_max - col_min
    if col_range > 0:
        lo_scaled = max(0, (lo_orig - col_min) / col_range)
        hi_scaled = min(1, (hi_orig - col_min) / col_range)
    else:
        lo_scaled, hi_scaled = 0.0, 1.0
    permitted_range[col] = [lo_scaled, hi_scaled]

print(f"\nPermitted ranges (scaled space, WHO/CCME guidelines):")
for col, (lo, hi) in permitted_range.items():
    lo_o, hi_o = permitted_range_original[col]
    print(f"  {col}: [{lo:.4f}, {hi:.4f}] (original: [{lo_o}, {hi_o}])")

print("\nDiCE wrapper setup complete!")

In [ ]:
# =============================================================================
# DiCE DATA PREPARATION
# =============================================================================

# Prepare training data with binary target for DiCE
y_train_binary = map_to_binary(y_train.values)
train_df_dice = X_train.copy().reset_index(drop=True)
train_df_dice['WaterQuality'] = y_train_binary

print(f"DiCE training data shape: {train_df_dice.shape}")
print(f"Binary distribution in training data:")
print(f"  Acceptable (0): {np.sum(y_train_binary == 0)}")
print(f"  Unacceptable (1): {np.sum(y_train_binary == 1)}")

# Create DiCE data interface
d = dice_ml.Data(
    dataframe=train_df_dice,
    continuous_features=feature_names,
    outcome_name='WaterQuality'
)

# Create DiCE model interface
m = dice_ml.Model(model=ngb_binary_wrapper, backend='sklearn', model_type='classifier')

# Create DiCE explainer
exp = dice_ml.Dice(d, m, method='random')
print("\nDiCE explainer created successfully!")
print("Method: random")
print("Target: Unacceptable (1) -> Acceptable (0)")

In [ ]:
# =============================================================================
# SAMPLE SELECTION FOR COUNTERFACTUALS
# Select samples predicted as Unacceptable (Marginal or Poor)
# =============================================================================

print("="*70)
print("SAMPLE SELECTION FOR COUNTERFACTUAL ANALYSIS")
print("="*70)

# Get predictions and probabilities
y_prob_test = ngb_binary_wrapper.predict_proba(X_test.values)

# Find Unacceptable predictions with various confidence levels
unacceptable_mask = y_binary_test == 1
unacceptable_indices = np.where(unacceptable_mask)[0]
unacceptable_probs = y_prob_test[unacceptable_mask, 1]  # P(Unacceptable)

print(f"Total test samples: {len(y_binary_test)}")
print(f"Predicted Unacceptable: {np.sum(unacceptable_mask)}")
print(f"Predicted Acceptable: {np.sum(~unacceptable_mask)}")

# Select 4 Unacceptable samples with different confidence levels
if len(unacceptable_indices) >= 4:
    sorted_idx = np.argsort(unacceptable_probs)[::-1]
    n_unacc = len(sorted_idx)
    selected_positions = [0, n_unacc//4, n_unacc//2, 3*n_unacc//4]
    selected_indices = [unacceptable_indices[sorted_idx[p]] for p in selected_positions]
else:
    selected_indices = list(unacceptable_indices[:4])

# Also select 4 Acceptable samples for sensitivity analysis
acceptable_mask = y_binary_test == 0
acceptable_indices_arr = np.where(acceptable_mask)[0]
acceptable_probs = y_prob_test[acceptable_mask, 0]

if len(acceptable_indices_arr) >= 4:
    sorted_idx_acc = np.argsort(acceptable_probs)
    n_acc = len(sorted_idx_acc)
    selected_positions_acc = [0, n_acc//4, n_acc//2, 3*n_acc//4]
    selected_indices_acc = [acceptable_indices_arr[sorted_idx_acc[p]] for p in selected_positions_acc]
else:
    selected_indices_acc = list(acceptable_indices_arr[:4])

all_selected = selected_indices + selected_indices_acc

print(f"\nSelected samples for counterfactual analysis:")
for i, idx in enumerate(all_selected):
    pred_5 = y_pred_test_5class[idx]
    pred_bin = y_binary_test[idx]
    prob_unacc = y_prob_test[idx, 1]
    status = "Unacceptable" if pred_bin == 1 else "Acceptable"
    print(f"  Sample {i+1} (idx={idx}): {class_names[pred_5]} -> {status} "
          f"(P(Unacceptable)={prob_unacc:.4f})")

In [ ]:
# =============================================================================
# GENERATE COUNTERFACTUAL EXPLANATIONS
# Target: Unacceptable -> Acceptable
# =============================================================================

print("="*70)
print("GENERATING COUNTERFACTUAL EXPLANATIONS")
print("Target: Unacceptable -> Acceptable")
print("="*70)

all_counterfactuals = []

for i, idx in enumerate(all_selected):
    pred_bin = y_binary_test[idx]
    pred_5 = y_pred_test_5class[idx]
    status = "Unacceptable" if pred_bin == 1 else "Acceptable"

    print(f"\nSample {i+1} (idx={idx}): {class_names[pred_5]} ({status})")

    query_instance = X_test.iloc[[idx]].reset_index(drop=True)

    try:
        cf = exp.generate_counterfactuals(
            query_instance,
            total_CFs=4,
            desired_class="opposite",
            permitted_range=permitted_range
        )
        all_counterfactuals.append(cf)
        print(f"  Generated {len(cf.cf_examples_list[0].final_cfs_df)} counterfactuals")
    except Exception as e:
        print(f"  Failed: {e}")
        all_counterfactuals.append(None)

print(f"\nTotal successful: {sum(1 for cf in all_counterfactuals if cf is not None)}/{len(all_selected)}")

### Evaluasi Constraint WHO/CCME

Evaluasi counterfactual terhadap standar kualitas air internasional. Constraint menggunakan WHO Guidelines for Drinking Water Quality (2022) dan Canadian Council of Ministers of the Environment (CCME) Water Quality Guidelines.

**Referensi:**
- WHO (2022) - Guidelines for Drinking Water Quality
- CCME - Canadian Water Quality Guidelines

In [ ]:
# =============================================================================
# EVALUASI CONSTRAINT - WHO/CCME Guidelines
# Referensi: WHO Guidelines for Drinking Water Quality (2022),
#            CCME Water Quality Guidelines
# =============================================================================

print("="*70)
print("EVALUASI CONSTRAINT WHO/CCME")
print("="*70)

# WHO/CCME constraints in original space
who_ccme_constraints = {
    'Ammonia (mg/l)': {'limit': 2.0, 'type': 'max', 'source': 'WHO'},
    'Biochemical Oxygen Demand (mg/l)': {'limit': 6.0, 'type': 'max', 'source': 'CCME'},
    'Dissolved Oxygen (mg/l)': {'limit': 4.0, 'type': 'min', 'source': 'CCME'},
    'Orthophosphate (mg/l)': {'limit': 1.0, 'type': 'max', 'source': 'CCME'},
    'pH (ph units)': {'limit_lo': 6.5, 'limit_hi': 8.5, 'type': 'range', 'source': 'WHO/CCME'},
    'Temperature (cel)': {'limit': 35.0, 'type': 'max', 'source': 'CCME'},
    'Nitrogen (mg/l)': {'limit': 10.0, 'type': 'max', 'source': 'WHO'},
    'Nitrate (mg/l)': {'limit': 50.0, 'type': 'max', 'source': 'WHO'}
}

print("\nWHO/CCME Water Quality Constraints:")
print(f"{'Parameter':<35} {'Constraint':<20} {'Source':<10}")
print("-"*65)
for param, info in who_ccme_constraints.items():
    if info['type'] == 'range':
        constraint_str = f"{info['limit_lo']} - {info['limit_hi']}"
    elif info['type'] == 'max':
        constraint_str = f"<= {info['limit']}"
    else:
        constraint_str = f">= {info['limit']}"
    print(f"{param:<35} {constraint_str:<20} {info['source']:<10}")

# Evaluate counterfactuals against constraints
print("\n" + "="*70)
print("EVALUASI COUNTERFACTUAL TERHADAP CONSTRAINT")
print("="*70)

for i, (idx, cf) in enumerate(zip(all_selected, all_counterfactuals)):
    if cf is None:
        continue

    pred_bin = y_binary_test[idx]
    if pred_bin != 1:
        continue  # Only evaluate Unacceptable -> Acceptable

    cf_df = cf.cf_examples_list[0].final_cfs_df
    if cf_df is None or len(cf_df) == 0:
        continue

    print(f"\nSample {i+1} (idx={idx}):")

    for cf_idx in range(min(2, len(cf_df))):
        cf_row = cf_df.iloc[cf_idx]
        print(f"  CF #{cf_idx+1}:")
        violations = 0
        for col in feature_names:
            col_i = feature_names.index(col)
            scaled_val = cf_row[col]
            # Convert to original space
            col_min = scaler.data_min_[col_i]
            col_max = scaler.data_max_[col_i]
            orig_val = scaled_val * (col_max - col_min) + col_min

            info = who_ccme_constraints[col]
            compliant = True
            if info['type'] == 'max' and orig_val > info['limit']:
                compliant = False
            elif info['type'] == 'min' and orig_val < info['limit']:
                compliant = False
            elif info['type'] == 'range':
                if orig_val < info['limit_lo'] or orig_val > info['limit_hi']:
                    compliant = False

            if not compliant:
                violations += 1
                print(f"    VIOLATION: {col} = {orig_val:.3f} ({info['source']})")

        if violations == 0:
            print(f"    All constraints satisfied!")
        else:
            print(f"    Total violations: {violations}")

### Rekomendasi Preskriptif

Langkah ini menghasilkan rekomendasi preskriptif berdasarkan hasil counterfactual analysis. Interpretasi output DiCE memberikan aksi spesifik yang perlu dilakukan untuk mengubah status kualitas air dari Unacceptable menjadi Acceptable, dengan mempertimbangkan constraint WHO/CCME.

**Referensi:**
- Mothilal et al. (2020) - DiCE: Diverse Counterfactual Explanations
- Lenatti et al. (2025) - Prescriptive recommendations from counterfactuals

In [ ]:
# =============================================================================
# REKOMENDASI PRESKRIPTIF
# =============================================================================

print("="*70)
print("ANALISIS REKOMENDASI PRESKRIPTIF (NGBoost + DiCE)")
print("="*70)
print("""
Interpretasi output DiCE:
- Sampel Unacceptable -> Acceptable = "Apa yang perlu DIPERBAIKI"
- Sampel Acceptable -> Unacceptable = "Parameter mana yang KRITIS"
- Rekomendasi di-evaluasi terhadap WHO/CCME Guidelines
""")

for i, (idx, cf) in enumerate(zip(all_selected, all_counterfactuals)):
    if cf is None:
        continue

    sample = X_test.iloc[idx]
    pred_5 = y_pred_test_5class[idx]
    pred_bin = y_binary_test[idx]
    prob_unacc = y_prob_test[idx, 1]

    status_awal = "Unacceptable" if pred_bin == 1 else "Acceptable"
    status_target = "Acceptable" if pred_bin == 1 else "Unacceptable"
    confidence = max(prob_unacc, 1 - prob_unacc) * 100

    if pred_bin == 1:
        jenis = "REKOMENDASI PERBAIKAN"
    else:
        jenis = "ANALISIS SENSITIVITAS"

    print(f"""
{'_'*60}
Sampel #{i+1} | {jenis}
{'_'*60}
""")
    print(f"  Kelas asli (5-class): {class_names[pred_5]}")
    print(f"  Status binary: {status_awal} (Confidence: {confidence:.1f}%)")
    print(f"  P(Unacceptable) = {prob_unacc:.4f}")

    if pred_bin == 1:
        print(f"  -> Rekomendasi: Ubah agar menjadi {status_target}")
    else:
        print(f"  -> Analisis: Parameter kritis yang menyebabkan perubahan status")

    # Ambil counterfactual pertama
    cf_df = cf.cf_examples_list[0].final_cfs_df
    if cf_df is not None and len(cf_df) > 0:
        cf_row = cf_df.iloc[0]
        changes = []
        for col in feature_names:
            col_idx = feature_names.index(col)
            original_scaled = sample[col]
            new_scaled = cf_row[col]

            if pd.notna(new_scaled) and abs(original_scaled - new_scaled) > 0.01:
                col_min = scaler.data_min_[col_idx]
                col_max = scaler.data_max_[col_idx]
                col_range = col_max - col_min
                orig_val = original_scaled * col_range + col_min
                new_val = new_scaled * col_range + col_min

                direction = "Naikkan" if new_val > orig_val else "Turunkan"
                changes.append({
                    'Parameter': col,
                    'Nilai Awal': f"{orig_val:.2f}",
                    'Nilai Target': f"{new_val:.2f}",
                    'Aksi': direction,
                    'Delta': abs(new_val - orig_val)
                })

        if changes:
            changes_df = pd.DataFrame(changes).sort_values('Delta', ascending=False)

            if pred_bin == 1:
                print(f"\n  Langkah perbaikan yang disarankan:")
            else:
                print(f"\n  Parameter kritis (jika berubah -> status berubah):")

            for j, (_, row) in enumerate(changes_df.iterrows(), 1):
                param = row['Parameter']
                comply = ""
                info = who_ccme_constraints[param]
                target_val = float(row['Nilai Target'])
                if info['type'] == 'max':
                    comply = " [WHO/CCME: OK]" if target_val <= info['limit'] else " [WHO/CCME: DILANGGAR]"
                elif info['type'] == 'min':
                    comply = " [WHO/CCME: OK]" if target_val >= info['limit'] else " [WHO/CCME: DILANGGAR]"
                elif info['type'] == 'range':
                    comply = " [WHO/CCME: OK]" if info['limit_lo'] <= target_val <= info['limit_hi'] else " [WHO/CCME: DILANGGAR]"

                print(f"     {j}. {row['Aksi']} {param}: "
                      f"{row['Nilai Awal']} -> {row['Nilai Target']}{comply}")

            print(f"\n  Jumlah parameter yang perlu diubah: {len(changes)}")
        else:
            print("  Tidak ada perubahan signifikan yang ditemukan.")
    print()

print("="*70)
print("RINGKASAN ANALISIS PRESKRIPTIF")
print("="*70)
print("""
TEMUAN UTAMA:
1. NGBoost memberikan PROBABILITAS per prediksi (bukan hanya label 5-class)
2. Binary grouping memungkinkan counterfactual yang interpretatif
3. DiCE memberikan REKOMENDASI SPESIFIK parameter yang perlu diubah
4. Constraint WHO/CCME memastikan rekomendasi FEASIBLE
5. Kombinasi NGBoost + DiCE = Probabilistic + Actionable Explanations
""")

In [ ]:
# =============================================================================
# EVALUASI COUNTERFACTUAL METRICS
# Referensi: Mothilal et al. (2020) - Validity, Proximity, Sparsity, Diversity
# =============================================================================

print("="*70)
print("EVALUASI KUALITAS COUNTERFACTUAL")
print("Referensi: Mothilal et al. (2020)")
print("="*70)

validity_scores = []
proximity_scores = []
sparsity_scores = []
diversity_scores = []
feasibility_scores = []

for i, (idx, cf) in enumerate(zip(all_selected, all_counterfactuals)):
    if cf is None:
        continue

    sample = X_test.iloc[idx].values
    pred_bin = y_binary_test[idx]
    desired = 1 - pred_bin

    cf_df = cf.cf_examples_list[0].final_cfs_df
    if cf_df is None or len(cf_df) == 0:
        continue

    # Validity: fraction of CFs that achieve desired class
    n_valid = 0
    cf_vectors = []
    for cf_idx in range(len(cf_df)):
        cf_row = cf_df.iloc[cf_idx][feature_names].values
        if np.any(pd.isna(cf_row)):
            continue
        cf_pred = ngb_binary_wrapper.predict(cf_row.reshape(1, -1))[0]
        if cf_pred == desired:
            n_valid += 1
        cf_vectors.append(cf_row)

    validity = n_valid / len(cf_df) if len(cf_df) > 0 else 0
    validity_scores.append(validity)

    # Proximity: average L1 distance
    if cf_vectors:
        distances = [np.mean(np.abs(cv - sample)) for cv in cf_vectors]
        proximity_scores.append(np.mean(distances))

    # Sparsity: average number of features changed
    if cf_vectors:
        n_changed = [np.sum(np.abs(cv - sample) > 0.01) for cv in cf_vectors]
        sparsity_scores.append(np.mean(n_changed))

    # Diversity: average pairwise distance between CFs
    if len(cf_vectors) > 1:
        pairwise_dists = []
        for a in range(len(cf_vectors)):
            for b in range(a+1, len(cf_vectors)):
                pairwise_dists.append(np.mean(np.abs(cf_vectors[a] - cf_vectors[b])))
        diversity_scores.append(np.mean(pairwise_dists))

    # Feasibility: fraction of CFs within WHO/CCME constraints
    if cf_vectors:
        n_feasible = 0
        for cv in cf_vectors:
            feasible = True
            for j, col in enumerate(feature_names):
                lo, hi = permitted_range[col]
                if cv[j] < lo - 0.01 or cv[j] > hi + 0.01:
                    feasible = False
                    break
            if feasible:
                n_feasible += 1
        feasibility_scores.append(n_feasible / len(cf_vectors))

# Print results
print(f"""
{'Metrik':<20} {'Mean':<15} {'Std':<15} {'Keterangan':<35}
{'-'*85}""")

if validity_scores:
    print(f"{'Validity':<20} {np.mean(validity_scores):<15.4f} {np.std(validity_scores):<15.4f} {'Fraksi CF yang valid':<35}")
if proximity_scores:
    print(f"{'Proximity':<20} {np.mean(proximity_scores):<15.4f} {np.std(proximity_scores):<15.4f} {'Jarak rata-rata (lebih kecil=baik)':<35}")
if sparsity_scores:
    print(f"{'Sparsity':<20} {np.mean(sparsity_scores):<15.4f} {np.std(sparsity_scores):<15.4f} {'Jumlah fitur diubah':<35}")
if diversity_scores:
    print(f"{'Diversity':<20} {np.mean(diversity_scores):<15.4f} {np.std(diversity_scores):<15.4f} {'Keragaman antar CF':<35}")
if feasibility_scores:
    print(f"{'Feasibility':<20} {np.mean(feasibility_scores):<15.4f} {np.std(feasibility_scores):<15.4f} {'Fraksi CF dalam constraint WHO/CCME':<35}")

# Interpretasi
print(f"""
{'='*70}
INTERPRETASI:
{'='*70}
""")
if validity_scores:
    val_status = "BAIK" if np.mean(validity_scores) > 0.8 else "MODERATE"
    print(f"- Validity {np.mean(validity_scores)*100:.1f}% ({val_status})")
if proximity_scores:
    prox_status = "BAIK (perubahan kecil)" if np.mean(proximity_scores) < 0.3 else "MODERATE"
    print(f"- Proximity {np.mean(proximity_scores):.4f} ({prox_status})")
if sparsity_scores:
    spar_status = "BAIK (sedikit fitur)" if np.mean(sparsity_scores) < 5 else "BANYAK fitur"
    print(f"- Sparsity {np.mean(sparsity_scores):.1f} fitur ({spar_status})")
if diversity_scores:
    print(f"- Diversity {np.mean(diversity_scores):.4f} (keragaman alternatif rekomendasi)")
if feasibility_scores:
    feas_status = "BAIK" if np.mean(feasibility_scores) > 0.8 else "PERLU PERHATIAN"
    print(f"- Feasibility {np.mean(feasibility_scores)*100:.1f}% ({feas_status})")

In [ ]:
# =============================================================================
# RINGKASAN HASIL EVALUASI
# =============================================================================

print("="*70)
print("RINGKASAN HASIL EVALUASI - Canada Water Quality (V2 Pipeline)")
print("="*70)
print(f"""
Metodologi:
- Preprocessing: Impute -> Scale -> Split 70/15/15 (Al Bataineh et al., 2026)
- SMOTE-ENN: {'Digunakan' if use_smote else 'Tidak digunakan (tidak meningkatkan performa)'}
- Hyperparameter: Optimized via Grid Search + 5-Fold CV (Nnadi et al., 2026)

Best Parameters:
- NGBoost: {best_ngb_params}
- XGBoost: {best_xgb_params}
- Random Forest: {best_rf_params}

Test Set Performance (F1 macro):
- NGBoost:       {results['NGBoost']['F1 (macro)']:.4f}
- XGBoost:       {results['XGBoost']['F1 (macro)']:.4f}
- Random Forest: {results['Random Forest']['F1 (macro)']:.4f}

5-Fold CV (F1 macro):
- NGBoost:       {np.mean(cv_results['NGBoost']):.4f} +/- {np.std(cv_results['NGBoost']):.4f}
- XGBoost:       {np.mean(cv_results['XGBoost']):.4f} +/- {np.std(cv_results['XGBoost']):.4f}
- Random Forest: {np.mean(cv_results['Random Forest']):.4f} +/- {np.std(cv_results['Random Forest']):.4f}
""")

## Summary & Conclusions

### Temuan Utama

1. **Preprocessing Pipeline**: Mengikuti Al Bataineh et al. (2026) Algorithm 3 - Impute, Scale, lalu Split 70/15/15
2. **SMOTE-ENN**: Dibandingkan secara kondisional; digunakan hanya jika meningkatkan performa
3. **Training NGBoost (Fixed Params)**: Baseline dengan parameter diagram (n_estimators=300, lr=0.05, minibatch_frac=0.8, col_sample=0.8, max_depth=4)
4. **Training Baseline Models**: XGBoost (multi:softprob) dan Random Forest dengan parameter default
5. **Hyperparameter Tuning**: Grid Search + 5-Fold CV mengikuti Nnadi et al. (2026)
6. **Evaluasi Before vs After**: Perbandingan metrik sebelum dan sesudah tuning
7. **Multi-class Classification**: NGBoost dengan k_categorical(5) untuk 5 kelas CCME WQI
8. **Binary Grouping for DiCE**: {Marginal, Poor} = Unacceptable, {Excellent, Good, Fair} = Acceptable
9. **Counterfactual Analysis**: Target Unacceptable -> Acceptable dengan constraint WHO/CCME
10. **Evaluasi Metrics**: Validity, Proximity, Sparsity, Diversity, Feasibility (Mothilal et al., 2020)

### Kontribusi Metodologis

| Aspek | Pendekatan |
|-------|-----------|
| Preprocessing | Al Bataineh et al. (2026) - Impute, Scale, Split |
| Class Imbalance | SMOTE-ENN kondisional (Zhu et al., 2023) |
| Initial Training | Parameter tetap dari diagram metodologi |
| Hyperparameter | Grid Search + 5-fold CV (Nnadi et al., 2026) |
| Probabilistic | NGBoost k_categorical(5) (Duan et al., 2020) |
| Kalibrasi | ECE + Uncertainty Zone (Guo et al., 2017) |
| Explainability | DiCE binary grouping (Mothilal et al., 2020) |
| Constraints | WHO/CCME guidelines for feasibility |

### Referensi

- Al Bataineh et al. (2026) - Algorithm 3: Preprocessing pipeline
- Patel et al. (2022) - Water potability dataset preprocessing
- Nnadi et al. (2026) - Grid search with 5-fold cross-validation
- Zhu et al. (2023) - SMOTE-ENN combined sampling method
- Duan et al. (2020) - NGBoost: Natural Gradient Boosting for Probabilistic Prediction
- Mothilal et al. (2020) - DiCE: Diverse Counterfactual Explanations
- Lenatti et al. (2025) - Multi-class counterfactual explanations
- Guo et al. (2017) - On Calibration of Modern Neural Networks
- Chen & Guestrin (2016) - XGBoost: A Scalable Tree Boosting System
- Breiman (2001) - Random Forests
- McNemar (1947) - Statistical comparison of classifiers
- WHO (2022) - Guidelines for Drinking Water Quality
- CCME - Canadian Water Quality Guidelines